# Extracción de datos judiciales del nicho familiar — LexData **v5**

**Mejoras v5 respecto a v4:**
1. 🔑 **Carga de credenciales desde `.env`** — `id_API` como `X-App-Token` (5 000 req/hora)
2. 🆕 **Fuente 7 — Fiscalía SPOA** (`6d52-qyqg`): Conteo Procesos V2, noticias criminales VIF
3. 🔁 **Retry con backoff exponencial** en `socrata_get` — resistente a endpoints inestables
4. 📊 **IVF normalizado per cápita** con proyecciones DANE 2024 (tasa por 100 000 hab.)
5. ⚖️  **IVF ponderado**: VIF 40 % · Alimentos 30 % · Medidas ICBF 20 % · Hurto 10 %
6. 🔒 **Pseudoanonimización SICOF** — SHA-256 + salt antes de persistir datos nominales

**Estado de fuentes (investigado abril 2025):**

| ID | Entidad | Estado |
|---|---|---|
| `ers2-kerr` | INMLCF VIF forense 2015-2024 | ✅ activo |
| `vuyt-mqpw` | Policía SIEDCO VIF (act. nov-2025) | ✅ activo |
| `4rxi-8m8d` | Policía SIEDCO hurto a personas | ✅ activo |
| `7tuu-upb2` | Min. Justicia comisarías Ley 2126 | ✅ activo |
| `6d52-qyqg` | Fiscalía Conteo Procesos V2 (SPOA) | ✅ **nuevo v5** |
| `wpqv-gzbz` | ICBF medidas protección | ⚠️ verificar |
| `x5yx-c7vy` | Rama Judicial procesos familia | ⚠️ verificar |

In [2]:
import requests
import pandas as pd
import time
import os
import re
import unicodedata
import hashlib
from urllib.parse import urljoin

# ── Cargar credenciales desde env/.env ───────────────────────────────────────
# El archivo debe tener: id_API=<tu_token>  y  clave_secreta_API=<tu_secret>
try:
    from dotenv import load_dotenv
    _candidates = [
        os.path.join("env", ".env"),                          # relativa al notebook
        os.path.join(os.path.dirname(os.getcwd()), "env", ".env"),  # un nivel arriba
        os.path.expanduser("~/Escritorio/Academico/data_thinking/env/.env"),
    ]
    for _path in _candidates:
        if os.path.exists(_path):
            load_dotenv(dotenv_path=_path)
            print(f"  .env cargado desde: {_path}")
            break
    else:
        print("  ⚠ .env no encontrado en rutas estándar — lee variables del SO")
except ImportError:
    print("  ⚠ python-dotenv no instalado.  pip install python-dotenv")

_APP_TOKEN = os.getenv("id_API", "")
_SECRET    = os.getenv("clave_secreta_API", "")

if _APP_TOKEN:
    print(f"✅ App Token activo: {_APP_TOKEN[:8]}… → 5 000 req/hora")
else:
    print("⚠️  Sin App Token — límite: 1 000 req/hora sin autenticar")
    print("   Registra uno en https://www.datos.gov.co/profile/app_tokens")

# ── Rutas y base URL ──────────────────────────────────────────────────────────
OUTPUT_DIR = "data_judicial"
BASE_URL   = "https://www.datos.gov.co"

# ── Catálogo de datasets (actualizado v5) ─────────────────────────────────────
DATASETS = {
    # VIF
    "vif_inmlcf":             "ers2-kerr",   # ✅ INMLCF forense 2015-2024
    "vif_policia":            "vuyt-mqpw",   # ✅ Policía SIEDCO VIF
    "vif_policia_ext":        "kmnf-h6r5",   # ✅ Policía VIF por municipio
    # Hurto
    "hurto_modalidades":      "d4fr-sbn2",   # ✅ Hurto por modalidades
    "hurto_personas":         "4rxi-8m8d",   # ✅ Hurto a personas
    # Directorio territorial
    "comisarias_directorio":  "7tuu-upb2",   # ✅ Comisarías Ley 2126
    # Fiscalía SPOA — NUEVO v5
    "fiscalia_procesos_vif":  "6d52-qyqg",   # ✅ Conteo Procesos V2 (Ley 906/1098)
    # Con posible migración de ID
    "procesos_rama_judicial": "x5yx-c7vy",   # ⚠️ Consejo Superior Judicatura
    "comisarias_familia_icbf":"wpqv-gzbz",   # ⚠️ ICBF medidas protección
}

BUSQUEDA_FALLBACK = {
    "procesos_rama_judicial":   "procesos judiciales especialidad familia rama",
    "comisarias_familia_icbf":  "comisarias familia medidas proteccion ICBF",
    "fiscalia_procesos_vif":    "conteo procesos SPOA violencia intrafamiliar fiscalia",
}

# ── Rango de años (v5: incluye 2025) ─────────────────────────────────────────
YEARS = list(range(2025, 2019, -1))

YEAR_COLS = ["a_o", "anio", "year", "a__o", "vigencia", "año"]

TIPOS_PROCESO_FAMILIAR = [
    "alimentos", "violencia intrafamiliar", "vif", "sustancias",
    "psicoactiv", "hurto", "patrimoni", "familia", "custodia",
    "filiacion", "paternidad", "medida de proteccion",
    "inasistencia alimentaria", "violencia dom",
]

DEPARTAMENTOS_FILTRO = ["VALLE DEL CAUCA"]
VALLE_BBOX = {"lat_min": 3.0, "lat_max": 5.5, "lon_min": -77.5, "lon_max": -75.5}

# ── Pesos IVF ponderado (v5) ──────────────────────────────────────────────────
# Justificación: VIF es el indicador más directo del ciclo familiar;
# hurto es un proxy más distal. Ajustar con comisarios en el piloto.
PESOS_IVF = {
    "vif_total":                0.40,
    "alimentos_familia_total":  0.30,
    "medidas_proteccion_total": 0.20,
    "hurto_total":              0.10,
}

# ── Proyecciones DANE 2024 — Valle del Cauca ──────────────────────────────────
# Fuente: DANE proyecciones CNPV 2018 (publicadas oct-2024)
# Usadas para normalizar el IVF a tasa por 100 000 hab.
DANE_POB_2024 = {
    "CALI":2237030, "PALMIRA":311063, "BUENAVENTURA":436665, "TULUA":221048,
    "JAMUNDI":167441, "YUMBO":118397, "GUADALAJARA DE BUGA":122601, "CANDELARIA":103840,
    "CARTAGO":138001, "FLORIDA":63458, "EL CERRITO":57248, "PRADERA":54283,
    "SEVILLA":44218, "ZARZAL":44100, "GUACARI":31420, "DAGUA":35800,
    "CALIMA":22100, "CAICEDONIA":28900, "BUGALAGRANDE":22000, "GINEBRA":20800,
    "LA UNION":36000, "ROLDANILLO":38000, "YOTOCO":17900, "ANDALUCIA":19800,
    "SAN PEDRO":16500, "ALCALA":17200, "LA CUMBRE":13100, "ANSERMANUEVO":16300,
    "RESTREPO":16000, "VIJES":14700, "RIOFRIO":17400, "OBANDO":13700,
    "TRUJILLO":17200, "LA VICTORIA":14000, "BOLIVAR":22100, "TORO":17900,
    "ULLOA":6800, "VERSALLES":8700, "EL DOVIO":9100, "EL AGUILA":9300,
    "EL CAIRO":8800, "ARGELIA":10400,
}

# ── Cabeceras HTTP ────────────────────────────────────────────────────────────
HEADERS = {
    "User-Agent": "Mozilla/5.0 (LexData-Scraper/5.0; nicho-familiar)",
    "Accept":     "application/json",
}
if _APP_TOKEN:
    HEADERS["X-App-Token"] = _APP_TOKEN   # activo automáticamente desde .env

print(f"\n✅ Config v5 | Años: {min(YEARS)}-{max(YEARS)} | Depto: {DEPARTAMENTOS_FILTRO}")

  .env cargado desde: env/.env
✅ App Token activo: 3wv7hh3o… → 5 000 req/hora

✅ Config v5 | Años: 2020-2025 | Depto: ['VALLE DEL CAUCA']


In [ ]:
# ─────────────────────────────────────────────────────────
# UTILIDADES COMUNES — v5 (retry + backoff exponencial)
# ─────────────────────────────────────────────────────────

def build_endpoint(dataset_id: str) -> str:
    return f"{BASE_URL}/resource/{dataset_id}.json"


def socrata_get(session, dataset_id, params, max_pages=50, max_retries=3):
    """
    GET paginado con retry + backoff exponencial (v5).
    Retorna lista vacía si el dataset no existe o falla persistentemente.
    """
    endpoint  = build_endpoint(dataset_id)
    PAGE_SIZE = 1000
    results, offset, page = [], 0, 0

    while page < max_pages:
        p = {**params, "$limit": PAGE_SIZE, "$offset": offset}
        batch = None

        for attempt in range(max_retries):
            try:
                r = session.get(endpoint, headers=HEADERS, params=p, timeout=30)
                if r.status_code == 404:
                    print(f"    ✗ Dataset {dataset_id} no encontrado (404).")
                    print(f"      → buscar_dataset_id(BUSQUEDA_FALLBACK.get('<clave>'))")
                    return []
                r.raise_for_status()
                batch = r.json()
                break
            except requests.exceptions.HTTPError as e:
                wait = 2 ** attempt
                print(f"    ⚠ HTTP {e} — reintento {attempt+1}/{max_retries} en {wait}s")
                time.sleep(wait)
            except requests.exceptions.ConnectionError:
                wait = 2 ** attempt
                print(f"    ⚠ Error de red — reintento {attempt+1}/{max_retries} en {wait}s")
                time.sleep(wait)
            except Exception as e:
                print(f"    ✗ Error inesperado: {e}")
                return results

        if batch is None:
            print(f"    ✗ {dataset_id} no responde — omitiendo, continuando pipeline.")
            break
        if not batch:
            break

        results.extend(batch)
        if len(batch) < PAGE_SIZE:
            break
        offset += PAGE_SIZE
        page   += 1
        time.sleep(0.4)

    return results


def contiene_tipo_familiar(texto):
    t = str(texto).lower()
    return any(kw in t for kw in TIPOS_PROCESO_FAMILIAR)


def filtrar_por_depto(registros, campo="departamento"):
    if not DEPARTAMENTOS_FILTRO:
        return registros
    deptos = [d.upper() for d in DEPARTAMENTOS_FILTRO]
    return [r for r in registros if str(r.get(campo, "")).upper() in deptos]


def safe_df(registros, fuente, dataset_id):
    df = pd.DataFrame(registros)
    if not df.empty:
        df["fuente"]      = fuente
        df["url_dataset"] = build_endpoint(dataset_id)
        df = df.drop_duplicates()
    return df


def normalizar_texto(t):
    """Mayúsculas sin tildes ni diacríticos para joins consistentes."""
    nfkd = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in nfkd if not unicodedata.combining(c)).upper().strip()


def pseudoanonimizar(valor: str, salt: str) -> str:
    """
    SHA-256 + salt para pseudoanonimizar identificadores personales (SICOF).
    El salt debe ser constante por organización y guardarse fuera de la BD.
    Cumple Ley 1581/2012 — los hashes no son datos personales identificables.

    Ejemplo de uso:
        hash_id = pseudoanonimizar("1234567890", salt="COMISARIA_CALI_2025")
    """
    if not valor or str(valor).strip() in ("", "nan", "None", "null"):
        return ""
    entrada = f"{salt}:{str(valor).strip()}".encode("utf-8")
    return hashlib.sha256(entrada).hexdigest()[:32]

SyntaxError: unexpected character after line continuation character (645173428.py, line 10)

In [3]:
# ─────────────────────────────────────────────────────────
# DIAGNÓSTICO Y REDESCUBRIMIENTO DE DATASETS
# ─────────────────────────────────────────────────────────

def inspect_dataset(session, dataset_id):
    """Obtiene 2 registros para confirmar disponibilidad y columnas."""
    endpoint = build_endpoint(dataset_id)
    try:
        r = session.get(endpoint, headers=HEADERS, params={"$limit": 2}, timeout=15)
        if r.status_code == 404:
            return {"disponible": False, "columnas": [], "muestra": []}
        r.raise_for_status()
        data = r.json()
        return {
            "disponible": True,
            "columnas": list(data[0].keys()) if data else [],
            "muestra": data,
        }
    except Exception as e:
        return {"disponible": False, "columnas": [], "muestra": [], "error": str(e)}


def buscar_dataset_id(termino, dominio="www.datos.gov.co", top_n=5):
    """
    Busca datasets en el catálogo de datos.gov.co usando la Socrata Discovery API.
    Útil cuando un ID retorna 404 y necesitas encontrar el nuevo identificador.

    Uso:
        buscar_dataset_id('procesos judiciales familia')
        buscar_dataset_id('violencia intrafamiliar ICBF comisarias')
    """
    url = "https://api.us.socrata.com/api/catalog/v1"
    params = {"q": termino, "domains": dominio, "limit": top_n}
    try:
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        results = r.json().get("results", [])
    except Exception as e:
        print(f"  ✗ Error en Discovery API: {e}")
        return

    if not results:
        print(f"  Sin resultados para: '{termino}'")
        print(f"  Busca manualmente en: https://{dominio}/browse?q={termino.replace(' ','+')}")
        return

    print(f"\n  Resultados para '{termino}':")
    print("  " + "─" * 62)
    for r in results:
        res = r.get("resource", {})
        uid  = res.get("id", "")
        name = res.get("name", "")[:56]
        dept = res.get("description", "")[:60]
        print(f"  ID: {uid}  →  {name}")
        if dept:
            print(f"       {dept}")
        print(f"       URL: {BASE_URL}/resource/{uid}.json")
    print()


# ── Ejecutar diagnóstico completo ─────────────────────────────────────────────
print("\n" + "═" * 62)
print("DIAGNÓSTICO DE DATASETS — LexData Nicho Familiar v4")
print("═" * 62)

session_diag = requests.Session()
DATASETS_DISPONIBLES = {}

for nombre, did in DATASETS.items():
    resultado = inspect_dataset(session_diag, did)
    DATASETS_DISPONIBLES[nombre] = resultado["disponible"]
    estado = "✅" if resultado["disponible"] else "❌"
    print(f"\n{estado} [{nombre}] → ID: {did}")
    if resultado["disponible"] and resultado["columnas"]:
        print(f"   Columnas ({len(resultado['columnas'])}): {', '.join(resultado['columnas'][:7])}")
        if resultado["muestra"]:
            for k, v in list(resultado["muestra"][0].items())[:3]:
                print(f"   · {k}: {str(v)[:45]}")
    elif not resultado["disponible"]:
        err = resultado.get("error", "404 Not Found")
        print(f"   ⚠ No disponible: {err}")
        if nombre in BUSQUEDA_FALLBACK:
            print(f"   → Buscando alternativas...")
            buscar_dataset_id(BUSQUEDA_FALLBACK[nombre])

print("\n" + "═" * 62)
activos = sum(DATASETS_DISPONIBLES.values())
print(f"Datasets activos: {activos}/{len(DATASETS)}")
if activos < len(DATASETS):
    print(f"⚠  {len(DATASETS) - activos} dataset(s) no disponibles.")
    print("   Los datos faltantes producirán DataFrames vacíos (pipeline no crashea).")
print("═" * 62)


══════════════════════════════════════════════════════════════
DIAGNÓSTICO DE DATASETS — LexData Nicho Familiar v4
══════════════════════════════════════════════════════════════

✅ [vif_inmlcf] → ID: ers2-kerr
   Columnas (36): id, a_o_del_hecho, sexo_de_la_victima, grupo_de_edad_quinquenal, grupo_mayor_menor_de_edad, grupo_de_edad_judicial, ciclo_vital
   · id: 1
   · a_o_del_hecho: 2015
   · sexo_de_la_victima: Hombre

✅ [vif_policia] → ID: vuyt-mqpw
   Columnas (8): departamento, municipio, codigo_dane, armas_medios, fecha_hecho, genero, grupo_etario
   · departamento: ANTIOQUIA
   · municipio: Medellín (CT)
   · codigo_dane: 05001000

✅ [vif_policia_ext] → ID: kmnf-h6r5
   Columnas (8): departamento, municipio, codigo_dane, armas_medios, fecha_hecho, genero, grupo_etario
   · departamento: BOYACÁ
   · municipio: Sogamoso
   · codigo_dane: 15759000

✅ [hurto_modalidades] → ID: d4fr-sbn2
   Columnas (9): departamento, municipio, codigo_dane, armas_medios, fecha_hecho, genero, grupo_

In [4]:
# ─────────────────────────────────────────────────────────
# FUENTE 1 — Rama Judicial (Consejo Superior de la Judicatura)
# Dataset: x5yx-c7vy  ⚠️  verificar disponibilidad en diagnóstico
# Columnas clave: especialidad, clase_proceso, anio, departamento, total_ingresos
# ─────────────────────────────────────────────────────────

def scrape_procesos_rama(session, years):
    DATASET_ID = DATASETS["procesos_rama_judicial"]

    if not DATASETS_DISPONIBLES.get("procesos_rama_judicial", True):
        print("  ⚠ Dataset procesos_rama_judicial no disponible — omitiendo.")
        print("  → Ejecuta buscar_dataset_id(BUSQUEDA_FALLBACK['procesos_rama_judicial'])")
        return pd.DataFrame()

    todos = []
    for year in years:
        print(f"  [Año {year}] Consultando procesos judiciales familiares...")
        where = (
            f"anio = '{year}' AND ("
            f"upper(especialidad) LIKE '%FAMILIA%' "
            f"OR upper(especialidad) LIKE '%PENAL%' "
            f"OR upper(clase_proceso) LIKE '%ALIMENTO%' "
            f"OR upper(clase_proceso) LIKE '%VIOLENCIA%')"
        )
        params = {
            "$where":  where,
            "$select": "especialidad,clase_proceso,anio,departamento,municipio,"
                       "total_ingresos,total_egresos,inventario_final",
            "$order":  "total_ingresos DESC",
        }
        raw = socrata_get(session, DATASET_ID, params)
        print(f"    → {len(raw)} registros obtenidos.")
        filtrados = filtrar_por_depto([
            r for r in raw
            if contiene_tipo_familiar(r.get("clase_proceso", ""))
            or contiene_tipo_familiar(r.get("especialidad", ""))
        ])
        print(f"    → {len(filtrados)} del nicho familiar retenidos.")
        todos.extend(filtrados)
        time.sleep(1.0)

    df = safe_df(todos, "Rama Judicial — Consejo Superior de la Judicatura", DATASET_ID)
    if not df.empty:
        df["tipo_ciclo"] = "ALIMENTOS_FAMILIA"
    print(f"\n  Total Rama Judicial: {len(df):,}")
    return df

In [5]:
# ─────────────────────────────────────────────────────────
# FUENTE 2 — ICBF Comisarías de Familia (Medidas de Protección)
# Dataset: wpqv-gzbz  ⚠️  verificar disponibilidad en diagnóstico
# Columnas clave: tipo_medida, tipo_violencia, anio, departamento, municipio, cantidad
# ─────────────────────────────────────────────────────────

def scrape_comisarias_icbf(session, years):
    DATASET_ID = DATASETS["comisarias_familia_icbf"]

    if not DATASETS_DISPONIBLES.get("comisarias_familia_icbf", True):
        print("  ⚠ Dataset comisarias_familia_icbf no disponible — omitiendo.")
        print("  → Ejecuta buscar_dataset_id(BUSQUEDA_FALLBACK['comisarias_familia_icbf'])")
        return pd.DataFrame()

    todos = []
    for year in years:
        print(f"  [Año {year}] Consultando ICBF medidas de protección...")
        where = f"anio = '{year}'"
        if DEPARTAMENTOS_FILTRO:
            deps = ", ".join([f"'{d.upper()}'" for d in DEPARTAMENTOS_FILTRO])
            where += f" AND upper(departamento) IN ({deps})"

        params = {
            "$where":  where,
            "$select": "tipo_medida,tipo_violencia,anio,departamento,municipio,"
                       "cantidad,sexo_victima,rango_edad_victima",
            "$order":  "cantidad DESC",
        }
        raw = socrata_get(session, DATASET_ID, params)
        print(f"    → {len(raw)} registros.")
        todos.extend(raw)
        time.sleep(1.0)

    df = safe_df(todos, "ICBF — Comisarías de Familia (Medidas de Protección)", DATASET_ID)
    if not df.empty:
        df["tipo_ciclo"] = "VIF"
        if "cantidad" in df.columns:
            df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0).astype(int)
    print(f"\n  Total ICBF Comisarías: {len(df):,}")
    return df

In [6]:
# ─────────────────────────────────────────────────────────
# FUENTE 3 — INMLCF Violencia Intrafamiliar (Cifras Forenses)
# Dataset: ers2-kerr  ✅  confirmado activo
# Columnas clave: a_o, departamento, municipio, sexo_de_la_victima
# ─────────────────────────────────────────────────────────

def scrape_vif_inmlcf(session, years):
    DATASET_ID = DATASETS["vif_inmlcf"]

    info = inspect_dataset(session, DATASET_ID)
    if not info["disponible"]:
        print("  ✗ Dataset vif_inmlcf no disponible.")
        return pd.DataFrame()

    # Detectar columna de año dinámicamente
    col_anio = next(
        (c for c in info["columnas"] if c.lower() in [y.lower() for y in YEAR_COLS]),
        None
    )
    print(f"  Columna de año detectada: '{col_anio}' | Columnas: {info['columnas'][:6]}")

    params = {}
    if DEPARTAMENTOS_FILTRO:
        deps = ", ".join([f"'{d.upper()}'" for d in DEPARTAMENTOS_FILTRO])
        params["$where"] = f"upper(departamento) IN ({deps})"

    raw = socrata_get(session, DATASET_ID, params)
    print(f"    → {len(raw)} registros totales obtenidos.")

    if col_anio:
        years_str = [str(y) for y in years]
        raw = [r for r in raw if str(r.get(col_anio, "")) in years_str]
        print(f"    → {len(raw)} dentro del rango {min(years)}–{max(years)}.")

    df = safe_df(raw, "INMLCF — Violencia Intrafamiliar Forense", DATASET_ID)
    if not df.empty:
        df["tipo_ciclo"] = "VIF"
        # Normalizar municipio para la tabla IVF
        col_mun = next((c for c in df.columns if "municipio" in c.lower()), None)
        if col_mun:
            df["municipio_norm"] = df[col_mun].apply(normalizar_texto)
    print(f"\n  Total VIF INMLCF: {len(df):,}")
    return df

In [7]:
# ─────────────────────────────────────────────────────────
# FUENTE 4 — Policía Nacional SIEDCO (Hurto)
# Datasets: d4fr-sbn2 (modalidades) ✅ + 4rxi-8m8d (personas) ✅
# Columnas clave: departamento, municipio, fecha_hecho, cantidad
# ─────────────────────────────────────────────────────────

def scrape_hurto_policia(session, years):
    sources = [
        ("hurto_modalidades", "Hurto por Modalidades"),
        ("hurto_personas",    "Hurto a Personas"),
    ]
    dfs = []

    for clave, label in sources:
        DATASET_ID = DATASETS[clave]
        print(f"  Consultando {label} ({DATASET_ID})...")

        info = inspect_dataset(session, DATASET_ID)
        if not info["disponible"]:
            print(f"  ✗ Dataset {clave} no disponible — omitiendo.")
            continue

        cols = info["columnas"]
        where_parts = []

        if DEPARTAMENTOS_FILTRO and "departamento" in cols:
            deps = ", ".join([f"'{d.upper()}'" for d in DEPARTAMENTOS_FILTRO])
            where_parts.append(f"upper(departamento) IN ({deps})")

        if "fecha_hecho" in cols and years:
            y_min, y_max = min(years), max(years)
            where_parts.append(
                f"fecha_hecho >= '{y_min}-01-01T00:00:00' "
                f"AND fecha_hecho <= '{y_max}-12-31T23:59:59'"
            )

        params = {"$where": " AND ".join(where_parts)} if where_parts else {}
        raw = socrata_get(session, DATASET_ID, params)
        print(f"    → {len(raw)} registros.")

        if raw:
            df_src = safe_df(raw, f"Policía Nacional SIEDCO — {label}", DATASET_ID)
            df_src["tipo_ciclo"] = "HURTO"
            df_src["sub_fuente"] = label
            dfs.append(df_src)

        time.sleep(1.0)

    df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    print(f"\n  Total Hurto Policía: {len(df):,}")
    return df

In [8]:
# ─────────────────────────────────────────────────────────
# FUENTE 5 — Directorio Comisarías de Familia (Min. Justicia)
# Dataset: 7tuu-upb2  ✅  confirmado activo
# Columnas clave: nombre_comisaria, departamento, municipio,
#                 latitud_de_ubicaci_n_de, longitud_de_ubicaci_n_de,
#                 coordenadas_de_ubicaci_n (DMS)
# ─────────────────────────────────────────────────────────

def scrape_comisarias_directorio(session):
    DATASET_ID = DATASETS["comisarias_directorio"]
    print(f"  Consultando directorio comisarías ({DATASET_ID})...")

    raw = socrata_get(session, DATASET_ID, {})

    # Filtro local: la columna de departamento en este dataset es 'nombre'
    # (nivel departamento) — filtramos por ambas posibilidades
    raw_filtrado = filtrar_por_depto(raw, campo="nombre")
    if not raw_filtrado:
        raw_filtrado = filtrar_por_depto(raw, campo="departamento")
    if not raw_filtrado:
        # Si no hay match de depto, revisar columna c_digo_dane_departamento
        raw_filtrado = [
            r for r in raw
            if str(r.get("c_digo_dane_departamento", "")).startswith("76")
            # DANE 76 = Valle del Cauca
        ]
    if not raw_filtrado:
        raw_filtrado = raw  # traer todos si no se puede filtrar

    print(f"    → {len(raw_filtrado)} comisarías obtenidas.")
    df = safe_df(raw_filtrado, "Ministerio de Justicia — Directorio Comisarías Ley 2126", DATASET_ID)
    if not df.empty:
        df["tipo_ciclo"] = "COMISARIA_GEO"
    print(f"\n  Total comisarías: {len(df):,}")
    return df

In [9]:
# ─────────────────────────────────────────────────────────
# FUENTE 6 — Policía Nacional SIEDCO (VIF Denuncias)
# Datasets: vuyt-mqpw ✅ + kmnf-h6r5 ✅
# Columnas clave: departamento, municipio, fecha_hecho, cantidad
# ─────────────────────────────────────────────────────────

def scrape_vif_policia(session, years):
    sources = [
        ("vif_policia",     "VIF Policía Nacional"),
        ("vif_policia_ext", "VIF por Municipio"),
    ]
    dfs = []

    for clave, label in sources:
        DATASET_ID = DATASETS[clave]
        print(f"  Consultando {label} ({DATASET_ID})...")

        info = inspect_dataset(session, DATASET_ID)
        if not info["disponible"]:
            print(f"  ✗ {clave} no disponible — omitiendo.")
            continue

        cols = info["columnas"]
        where_parts = []

        if DEPARTAMENTOS_FILTRO and "departamento" in cols:
            deps = ", ".join([f"'{d.upper()}'" for d in DEPARTAMENTOS_FILTRO])
            where_parts.append(f"upper(departamento) IN ({deps})")

        if "fecha_hecho" in cols and years:
            y_min, y_max = min(years), max(years)
            where_parts.append(
                f"fecha_hecho >= '{y_min}-01-01T00:00:00' "
                f"AND fecha_hecho <= '{y_max}-12-31T23:59:59'"
            )

        params = {"$where": " AND ".join(where_parts), "$order": "fecha_hecho DESC"} if where_parts else {"$order": "fecha_hecho DESC"}
        raw = socrata_get(session, DATASET_ID, params)
        print(f"    → {len(raw)} registros.")

        if raw:
            df_src = safe_df(raw, f"Policía Nacional SIEDCO — {label}", DATASET_ID)
            df_src["tipo_ciclo"] = "VIF"
            df_src["sub_fuente"] = label
            if "cantidad" in df_src.columns:
                df_src["cantidad"] = pd.to_numeric(df_src["cantidad"], errors="coerce")
            dfs.append(df_src)

        time.sleep(1.0)

    df = pd.concat(dfs, ignore_index=True).drop_duplicates() if dfs else pd.DataFrame()
    print(f"\n  Total VIF Policía: {len(df):,}")
    return df

---
## Fuente 7 — Fiscalía SPOA: Conteo Procesos V2 *(nuevo en v5)*

Dataset `6d52-qyqg` publicado por la Fiscalía General de la Nación.
Contiene noticias criminales del Sistema Penal Oral Acusatorio (Ley 906/2004 y 1098/2006) desde 2010.

**Filtro aplicado:** `grupo_delito LIKE '%VIOLENCIA%'` + departamento Valle del Cauca.
Columnas clave: `departamento`, `municipio`, `anio_entrada`, `grupo_delito`, `total_procesos`.

In [ ]:
# ─────────────────────────────────────────────────────────
# FUENTE 7 — Fiscalía SPOA (NUEVO v5)
# Dataset: 6d52-qyqg  ✅  Conteo de Procesos V2 (Ley 906 + 1098)
# Columnas: departamento, municipio, anio_entrada,
#           grupo_delito, nombre_delito, total_procesos
# ─────────────────────────────────────────────────────────

def scrape_fiscalia_vif(session, years):
    DATASET_ID = DATASETS["fiscalia_procesos_vif"]

    if not DATASETS_DISPONIBLES.get("fiscalia_procesos_vif", True):
        print("  ⚠ fiscalia_procesos_vif no disponible — omitiendo.")
        print("  → Ejecuta: buscar_dataset_id(BUSQUEDA_FALLBACK['fiscalia_procesos_vif'])")
        return pd.DataFrame()

    todos = []
    for year in years:
        print(f"  [Año {year}] Fiscalía SPOA VIF...")
        where_parts = ["upper(grupo_delito) LIKE '%VIOLENCIA%'"]
        if DEPARTAMENTOS_FILTRO:
            deps = ", ".join([f"'{d.upper()}'" for d in DEPARTAMENTOS_FILTRO])
            where_parts.append(f"upper(departamento) IN ({deps})")
        where_parts.append(f"anio_entrada = '{year}'")

        params = {
            "$where":  " AND ".join(where_parts),
            "$select": "departamento,municipio,anio_entrada,grupo_delito,"
                       "nombre_delito,total_procesos",
            "$order":  "total_procesos DESC",
        }
        raw = socrata_get(session, DATASET_ID, params)

        # Si no hay resultados con año exacto, intentar sin ese filtro
        if not raw:
            params2 = {"$where": " AND ".join(where_parts[:-1]),
                       "$select": params["$select"]}
            raw = socrata_get(session, DATASET_ID, params2)
            raw = [r for r in raw if str(r.get("anio_entrada","")) == str(year)]

        print(f"    → {len(raw)} registros.")
        todos.extend(raw)
        time.sleep(1.0)

    df = safe_df(todos, "Fiscalía — Conteo Procesos V2 SPOA", DATASET_ID)
    if not df.empty:
        df["tipo_ciclo"] = "VIF"
        df["sub_fuente"] = "Fiscalia_SPOA"
        if "total_procesos" in df.columns:
            df["total_procesos"] = pd.to_numeric(df["total_procesos"], errors="coerce").fillna(0).astype(int)
        if "municipio" in df.columns:
            df["municipio_norm"] = df["municipio"].apply(normalizar_texto)
        if "anio_entrada" in df.columns:
            df = df.rename(columns={"anio_entrada": "anio"})
    print(f"\n  Total Fiscalía VIF: {len(df):,}")
    return df

---
## Módulo SICOF — Ingesta de datos privados nominales

> **Por qué este módulo existe:**  
> Los datos de `datos.gov.co` son estadísticas *agregadas* por municipio y mes.  
> Para vincular expedientes del mismo núcleo familiar se necesitan datos *nominales*  
> (nombres, documentos, direcciones) que solo las entidades piloto pueden proveer directamente.
>
> **Separación macro / micro:**
> - **Macro** → datos públicos Socrata: contexto territorial, líneas base, ponderaciones IVF  
> - **Micro** → exportaciones SICOF privadas: vinculación familiar real, entrenamiento del motor

**Para activar:** solicitar a la entidad piloto (comisaría, juzgado) una exportación CSV o Excel  
de sus casos activos con las columnas del esquema indicado abajo, y pasar la ruta a `ingestar_sicof()`.

In [10]:
# ─────────────────────────────────────────────────────────
# MÓDULO SICOF — DATOS NOMINALES PRIVADOS + PSEUDOANONIMIZACIÓN (v5)
# ─────────────────────────────────────────────────────────

# ── Salt por organización (v5) ────────────────────────────────────────────────
# Cargar desde variable de entorno. NUNCA hardcodear en producción.
_SALT_SICOF = os.getenv("SALT_SICOF_ORG", "LEXDATA_PILOTO_CALI_2025")


def ingestar_sicof(ruta_csv: str, salt: str = _SALT_SICOF) -> pd.DataFrame:
    """
    Carga exportación SICOF y pseudoanonimiza identificadores personales.
    - cédula y nombre → hashes SHA-256+salt (128 bits)
    - municipio y tipo_proceso se conservan en claro para analítica
    - Cumple Ley 1581/2012 — los hashes no son PII identificable directamente
    """
    if not os.path.exists(ruta_csv):
        print(f"  ✗ Archivo SICOF no encontrado: {ruta_csv}")
        return pd.DataFrame()

    df = pd.read_csv(ruta_csv, dtype=str).fillna("")

    # Pseudoanonimizar campos sensibles
    for col_personal, col_hash in [("cedula_parte","hash_cedula"),("nombre_parte","hash_nombre")]:
        if col_personal in df.columns:
            df[col_hash] = df[col_personal].apply(lambda v: pseudoanonimizar(v, salt))
            df = df.drop(columns=[col_personal])
            print(f"  🔒 {col_personal} → {col_hash} (SHA-256+salt)")

    if "municipio" in df.columns:
        df["municipio_norm"] = df["municipio"].apply(normalizar_texto)
    if "fecha_radicado" in df.columns:
        df["fecha_radicado"] = pd.to_datetime(df["fecha_radicado"], errors="coerce")
        df["anio"]      = df["fecha_radicado"].dt.year
        df["mes"]       = df["fecha_radicado"].dt.month
        df["trimestre"] = df["fecha_radicado"].dt.quarter

    df["fuente"]     = "SICOF — Exportación Privada Piloto"
    df["tipo_ciclo"] = "SICOF_PRIVADO"
    print(f"  ✅ SICOF cargado: {len(df):,} registros pseudoanonimizados")
    print(f"     Salt: {salt[:12]}… (guardar fuera de la BD)")
    return df


def vincular_nucleo_familiar(df_sicof: pd.DataFrame) -> pd.DataFrame:
    """
    Detecta núcleos familiares: personas que aparecen como parte
    en múltiples tipos de proceso en el mismo municipio.
    Retorna df con columna id_nucleo para cruzar con tabla IVF.
    """
    if df_sicof.empty:
        return df_sicof
    df = df_sicof.copy()

    if "hash_cedula" in df.columns and "municipio_norm" in df.columns:
        conteos = (
            df.groupby(["hash_cedula","municipio_norm"])
              .size()
              .reset_index(name="n_expedientes")
        )
        df = df.merge(conteos, on=["hash_cedula","municipio_norm"], how="left")
        df["id_nucleo"] = (df["municipio_norm"].astype(str) + "_" +
                           df["hash_cedula"].astype(str).str[:8])
        nucleos_multi = (df["n_expedientes"] > 1).sum()
        print(f"  Registros en múltiples expedientes: {nucleos_multi:,}")
        print(f"  Núcleos únicos detectados: {df['id_nucleo'].nunique():,}")
    return df


# ── Ejemplo de uso (descomentar cuando tengas el CSV del piloto) ──────────────
# df_sicof = ingestar_sicof("data_judicial/exportacion_sicof_piloto.csv")
# df_sicof = vincular_nucleo_familiar(df_sicof)

MÓDULO SICOF — INSTRUCCIONES

  Para activar con datos reales de la comisaría piloto:
  ─────────────────────────────────────────────────────
  df_sicof    = ingestar_sicof('ruta/comisaria_cali_2024.csv')
  df_vinculado = vincular_nucleo_familiar(df_sicof)

  Columnas esperadas en la exportación:
    radicado                       str          ✦ obligatoria
    fecha_radicacion               datetime     ✦ obligatoria
    tipo_proceso                   str          ✦ obligatoria
    nombre_demandante              str            opcional
    nombre_demandado               str            opcional
    documento_demandante           str            opcional
    documento_demandado            str            opcional
    municipio                      str          ✦ obligatoria
    direccion                      str            opcional
    estado                         str            opcional
    juez_despacho                  str            opcional

  ⚠  CONFIDENCIAL: estos datos nunca deb

In [11]:
# ─────────────────────────────────────────────────────────
# ORQUESTADOR — consolida las SIETE fuentes públicas (v5)
# ─────────────────────────────────────────────────────────

NOMBRES_ARCHIVO = {
    "rama_judicial":         "lexdata_rama_judicial.csv",
    "comisarias_icbf":       "lexdata_comisarias_icbf.csv",
    "vif_inmlcf":            "lexdata_vif_inmlcf.csv",
    "hurto_policia":         "lexdata_hurto_policia.csv",
    "comisarias_directorio": "lexdata_comisarias_directorio_geo.csv",
    "vif_policia":           "lexdata_vif_policia.csv",
    "fiscalia_vif":          "lexdata_fiscalia_vif.csv",      # nuevo v5
}


def scrape_nicho_familiar(years):
    session = requests.Session()
    print("\n" + "=" * 62)
    print("SCRAPING v5 — LexData Nicho Familiar")
    print("=" * 62)

    resultados = {}

    print("\n[1/7] Rama Judicial — Consejo Superior de la Judicatura")
    resultados["rama_judicial"] = scrape_procesos_rama(session, years)

    print("\n[2/7] ICBF — Comisarías de Familia")
    resultados["comisarias_icbf"] = scrape_comisarias_icbf(session, years)

    print("\n[3/7] INMLCF — VIF Forense")
    resultados["vif_inmlcf"] = scrape_vif_inmlcf(session, years)

    print("\n[4/7] Policía SIEDCO — Hurto")
    resultados["hurto_policia"] = scrape_hurto_policia(session, years)

    print("\n[5/7] Min. Justicia — Directorio Comisarías")
    resultados["comisarias_directorio"] = scrape_comisarias_directorio(session)

    print("\n[6/7] Policía SIEDCO — VIF Denuncias")
    resultados["vif_policia"] = scrape_vif_policia(session, years)

    print("\n[7/7] Fiscalía — Conteo Procesos V2 SPOA")
    resultados["fiscalia_vif"] = scrape_fiscalia_vif(session, years)

    print("\n" + "=" * 62)
    print("Scraping completado.")
    return resultados

In [12]:
# ── Ejecutar scraping ─────────────────────────────────────────────────────────
datos = scrape_nicho_familiar(years=YEARS)

# ── Exportar CSVs individuales ────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)

for clave, df in datos.items():
    if df.empty:
        print(f"⚠️  [{clave}] vacío — dataset no disponible o sin registros.")
        continue
    ruta = os.path.join(OUTPUT_DIR, NOMBRES_ARCHIVO[clave])
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"✅ [{clave}] → {ruta}  ({len(df):,} filas)")

# ── Consolidado transaccional ─────────────────────────────────────────────────
EXCLUIR = {"comisarias_directorio"}
dfs_tx = [df for k, df in datos.items() if not df.empty and k not in EXCLUIR]

if dfs_tx:
    df_consolidado = pd.concat(dfs_tx, ignore_index=True)
    ruta_cons = os.path.join(OUTPUT_DIR, "lexdata_nicho_familiar_CONSOLIDADO.csv")
    df_consolidado.to_csv(ruta_cons, index=False, encoding="utf-8-sig")
    fuentes_activas = [k for k in datos if k not in EXCLUIR and not datos[k].empty]
    print(f"\n✅ Consolidado → {ruta_cons}  ({len(df_consolidado):,} filas)")
    print(f"   Fuentes incluidas: {fuentes_activas}")
else:
    print("\n⚠️  Consolidado vacío — todos los DataFrames transaccionales están vacíos.")


FUENTE 1 — Rama Judicial
  ⚠ Dataset procesos_rama_judicial no disponible — omitiendo.
  → Ejecuta buscar_dataset_id(BUSQUEDA_FALLBACK['procesos_rama_judicial'])

FUENTE 2 — ICBF Comisarías (Medidas de Protección)
  ⚠ Dataset comisarias_familia_icbf no disponible — omitiendo.
  → Ejecuta buscar_dataset_id(BUSQUEDA_FALLBACK['comisarias_familia_icbf'])

FUENTE 3 — INMLCF VIF Forense
  Columna de año detectada: 'None' | Columnas: ['id', 'a_o_del_hecho', 'sexo_de_la_victima', 'grupo_de_edad_quinquenal', 'grupo_mayor_menor_de_edad', 'grupo_de_edad_judicial']
    ✗ HTTP 400 Client Error: Bad Request for url: https://www.datos.gov.co/resource/ers2-kerr.json?%24where=upper%28departamento%29+IN+%28%27VALLE+DEL+CAUCA%27%29&%24limit=1000&%24offset=0
    → 0 registros totales obtenidos.

  Total VIF INMLCF: 0

FUENTE 4 — Policía SIEDCO Hurto
  Consultando Hurto por Modalidades (d4fr-sbn2)...
    → 0 registros.
  Consultando Hurto a Personas (4rxi-8m8d)...
    → 17408 registros.

  Total Hurto Pol

In [13]:
# ── Vista previa por fuente ───────────────────────────────────────────────────
for clave, df in datos.items():
    if df.empty:
        print(f"\n[{clave.upper()}] → sin datos")
        continue
    print(f"\n{'─'*62}")
    print(f"  {clave.upper().replace('_',' ')}  ({len(df):,} registros)")
    print(f"  Columnas: {list(df.columns[:8])}")
    print(f"{'─'*62}")
    print(df.head(3).to_string(index=False, max_colwidth=35))


[RAMA_JUDICIAL] → sin datos

[COMISARIAS_ICBF] → sin datos

[VIF_INMLCF] → sin datos

──────────────────────────────────────────────────────────────
  HURTO POLICIA  (17,408 registros)
  Columnas: ['fecha_hecho', 'cod_depto', 'departamento', 'cod_muni', 'municipio', 'cantidad', 'fuente', 'url_dataset']
──────────────────────────────────────────────────────────────
            fecha_hecho cod_depto    departamento cod_muni municipio cantidad                              fuente                         url_dataset tipo_ciclo       sub_fuente
2020-01-01T00:00:00.000        76 VALLE DEL CAUCA    76275   FLORIDA        1 Policía Nacional SIEDCO — Hurto ... https://www.datos.gov.co/resourc...      HURTO Hurto a Personas
2020-01-01T00:00:00.000        76 VALLE DEL CAUCA    76736   SEVILLA        1 Policía Nacional SIEDCO — Hurto ... https://www.datos.gov.co/resourc...      HURTO Hurto a Personas
2020-01-01T00:00:00.000        76 VALLE DEL CAUCA    76147   CARTAGO        2 Policía Nacional SIE

---
## Limpieza y estandarización de datos

Correcciones aplicadas (v4):

**Coordenadas geoespaciales (comisarías):**
1. **Estrategia 1 — DMS:** parsea la columna `coordenadas_de_ubicaci_n` en formato `3°36'52.8"N 76°23'16.1"W`
2. **Estrategia 2 — Enteros escalonados:** detecta formatos como `-76,387,812` (= -76.387812) y `3548598` (= 3.548598) dividiendo por potencias de 10 hasta obtener un valor WGS84 válido
3. **Validación geográfica:** rechaza coordenadas fuera del bounding box del Valle del Cauca (lat 3.0–5.5°N, lon -77.5 a -75.5°W)

In [14]:
# ─────────────────────────────────────────────────────────
# LIMPIEZA Y ESTANDARIZACIÓN — v4
# ─────────────────────────────────────────────────────────

def dms_to_decimal(s):
    """
    Convierte formato sexagesimal DMS a coordenadas decimales WGS84.

    Estrategia sin regex complejo:
      Normaliza los delimitadores (°, ', ", ′, ″) a espacios,
      luego extrae grados/minutos/segundos y la letra de dirección.

    Ejemplos:
      '3°36\'52.8\"N 76°23\'16.1\"W'  → (3.614667, -76.387833)
      '3°31\'14.7\"N 76°17\'42.8\"W'  → (3.520750, -76.295222)
    """
    try:
        # Normalizar delimitadores a espacio
        clean = (
            str(s)
            .replace("°", " ")
            .replace("′", " ").replace("'", " ")
            .replace("″", " ").replace('"', " ")
        )
        tokens = clean.split()

        # Buscar posiciones de las letras de dirección
        dirs = [(i, t.upper()) for i, t in enumerate(tokens)
                if t.upper() in ("N", "S", "E", "W")]

        if len(dirs) < 2:
            return None, None

        i1, d1 = dirs[0]
        i2, d2 = dirs[1]

        lat_tok = tokens[:i1]
        lon_tok = tokens[i1 + 1:i2]

        if len(lat_tok) < 3 or len(lon_tok) < 3:
            return None, None

        lat = int(lat_tok[0]) + int(lat_tok[1]) / 60 + float(lat_tok[2]) / 3600
        lon = int(lon_tok[0]) + int(lon_tok[1]) / 60 + float(lon_tok[2]) / 3600

        if d1 == "S":
            lat = -lat
        if d2 == "W":
            lon = -lon

        return round(lat, 6), round(lon, 6)

    except (ValueError, IndexError):
        return None, None


def limpiar_coord_numerica(val):
    """
    Convierte coordenadas numéricas mal formateadas a float WGS84.

    Casos manejados:
      '-76,387,812' → comas como miles → -76387812 → ÷1e6 → -76.387812
      '3548598'     → entero puro      →   3548598 → ÷1e6 →   3.548598
      '-76.315400'  → ya es float válido
      '3,525,076.00'→ comas como miles →  3525076  → ÷1e6 →   3.525076
    """
    if pd.isna(val) or str(val).strip() in ("", "Null", "null", "None", "nan"):
        return None
    s = str(val).strip().replace(" ", "")
    n_comas = s.count(",")
    if n_comas > 1:
        s = s.replace(",", "")
    elif n_comas == 1:
        partes = s.split(",")
        if len(partes[1]) == 3 and partes[1].isdigit():
            s = s.replace(",", "")
        else:
            s = s.replace(",", ".")
    try:
        v = float(s)
    except ValueError:
        return None

    if abs(v) <= 180:
        return round(v, 6)

    # Escalar por potencias de 10 hasta obtener valor WGS84 válido
    for k in range(1, 8):
        scaled = v / (10 ** k)
        if abs(scaled) <= 180:
            return round(scaled, 6)

    return None


def en_valle_bbox(lat, lon):
    """Valida que las coordenadas estén dentro del bounding box del Valle del Cauca."""
    if lat is None or lon is None:
        return False
    return (
        VALLE_BBOX["lat_min"] <= lat <= VALLE_BBOX["lat_max"] and
        VALLE_BBOX["lon_min"] <= lon <= VALLE_BBOX["lon_max"]
    )


def limpiar_comisarias(df):
    """
    Georreferencia el DataFrame de comisarías con tres estrategias en cascada:
      1. DMS → columna coordenadas_de_ubicaci_n
      2. Numérico escalado → columnas latitud / longitud
      3. Validación bounding box Valle del Cauca
    """
    df = df.copy()
    n_total = len(df)

    col_dms = next((c for c in df.columns if "coordenada" in c.lower()), None)
    col_lat = next((c for c in df.columns if "latitud" in c.lower()), None)
    col_lon = next((c for c in df.columns if "longitud" in c.lower()), None)

    df["lat"] = None
    df["lon"] = None
    rechazadas_bbox = 0

    for idx, row in df.iterrows():
        lat, lon = None, None

        # Estrategia 1: DMS
        if col_dms and pd.notna(row.get(col_dms)):
            lat, lon = dms_to_decimal(str(row[col_dms]))

        # Estrategia 2: numérico escalado (fallback)
        if lat is None and col_lat:
            lat = limpiar_coord_numerica(row.get(col_lat))
        if lon is None and col_lon:
            lon = limpiar_coord_numerica(row.get(col_lon))

        # Estrategia 3: bounding box Valle del Cauca
        if lat is not None and lon is not None:
            if not en_valle_bbox(lat, lon):
                rechazadas_bbox += 1
                nombre = row.get("nombre_comisaria", row.get("nombre", "?"))
                print(f"  ⚠ Fuera del Valle del Cauca → descartada "
                      f"({str(nombre)[:40]}): lat={lat}, lon={lon}")
                lat, lon = None, None

        df.at[idx, "lat"] = lat
        df.at[idx, "lon"] = lon

    # Normalizar nombres
    for col_orig, col_norm in [("nombre_1", "municipio_norm"), ("nombre", "departamento_norm")]:
        if col_orig in df.columns:
            df[col_norm] = df[col_orig].apply(normalizar_texto)

    n_geo = df[["lat", "lon"]].notna().all(axis=1).sum()
    print(f"\n  Resumen georreferenciación:")
    print(f"    Total comisarías:                     {n_total}")
    print(f"    Con coordenadas válidas (WGS84):      {n_geo}")
    print(f"    Descartadas (fuera del bounding box): {rechazadas_bbox}")
    print(f"    Sin coordenadas en fuente original:   {n_total - n_geo - rechazadas_bbox}")

    return df


def enriquecer_temporal(df, col_fecha="fecha_hecho"):
    df = df.copy()
    if col_fecha not in df.columns:
        return df
    df[col_fecha] = pd.to_datetime(df[col_fecha], errors="coerce")
    df["anio"]      = df[col_fecha].dt.year
    df["mes"]       = df[col_fecha].dt.month
    df["trimestre"] = df[col_fecha].dt.quarter
    if "municipio" in df.columns:
        df["municipio_norm"] = df["municipio"].apply(
            lambda x: normalizar_texto(x) if pd.notna(x) else x
        )
    return df


# ── Aplicar limpieza ──────────────────────────────────────────────────────────
print("=" * 62)
print("LIMPIEZA Y ESTANDARIZACIÓN v4")
print("=" * 62)

datos_limpios = {}

for clave, df in datos.items():
    if df.empty:
        datos_limpios[clave] = df
        continue

    if clave == "comisarias_directorio":
        print(f"\n[{clave}] Georreferenciando con DMS + escalado numérico + bbox...")
        datos_limpios[clave] = limpiar_comisarias(df)
    elif "fecha_hecho" in df.columns:
        print(f"\n[{clave}] Enriqueciendo temporalmente...")
        datos_limpios[clave] = enriquecer_temporal(df)
        if "cantidad" in datos_limpios[clave].columns:
            datos_limpios[clave]["cantidad"] = (
                pd.to_numeric(datos_limpios[clave]["cantidad"], errors="coerce")
                .fillna(0).astype(int)
            )
    else:
        datos_limpios[clave] = df

    n = len(datos_limpios[clave])
    print(f"  [{clave}] → {n:,} filas | cols: {list(datos_limpios[clave].columns[:6])}")

# Re-exportar versiones limpias
os.makedirs(OUTPUT_DIR, exist_ok=True)
for clave, df in datos_limpios.items():
    if df.empty:
        continue
    ruta = os.path.join(OUTPUT_DIR, NOMBRES_ARCHIVO.get(clave, f"lexdata_{clave}.csv"))
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"✅ [{clave}] re-exportado → {ruta}")

print("\n✅ Limpieza v4 completada.")

LIMPIEZA Y ESTANDARIZACIÓN v4

[hurto_policia] Enriqueciendo temporalmente...
  [hurto_policia] → 17,408 filas | cols: ['fecha_hecho', 'cod_depto', 'departamento', 'cod_muni', 'municipio', 'cantidad']

[comisarias_directorio] Georreferenciando con DMS + escalado numérico + bbox...
  ⚠ Fuera del Valle del Cauca → descartada (cOMISARIA 1, 2, 3 Y 4): lat=5.340194, lon=-72.381778

  Resumen georreferenciación:
    Total comisarías:                     60
    Con coordenadas válidas (WGS84):      54
    Descartadas (fuera del bounding box): 1
    Sin coordenadas en fuente original:   5
  [comisarias_directorio] → 60 filas | cols: ['c_digo_dane_departamento', 'nombre', 'c_digo_dane_municipio', 'nombre_1', 'tipo_municipio_isla_rea_no', 'categoria_municipio']
✅ [hurto_policia] re-exportado → data_judicial/lexdata_hurto_policia.csv
✅ [comisarias_directorio] re-exportado → data_judicial/lexdata_comisarias_directorio_geo.csv

✅ Limpieza v4 completada.


---
## Tabla de co-ocurrencia — Motor IVF (4 dimensiones)

Construye la tabla que alimenta el entrenamiento del Motor IVF con las **cuatro dimensiones del ciclo familiar**:

| Dimensión | Fuente | Estado |
|-----------|--------|--------|
| `hurto_total` | Policía SIEDCO | ✅ datos disponibles |
| `vif_total` | Policía SIEDCO + INMLCF | ✅ datos disponibles |
| `medidas_proteccion_total` | ICBF Comisarías | ⚠️ depende de dataset |
| `alimentos_familia_total` | Rama Judicial | ⚠️ depende de dataset |

> Las columnas de datasets no disponibles aparecerán como `0` (no bloquean la tabla).  
> Activar el Módulo SICOF para añadir una quinta dimensión con datos nominales del piloto.

In [15]:
# ─────────────────────────────────────────────────────────
# TABLA IVF — 4 DIMS + NORMALIZACIÓN PER CÁPITA + PONDERACIÓN (v5)
# ─────────────────────────────────────────────────────────

def agrupar_municipio(df, col_cantidad=None, label=None):
    """Agrupa por municipio_norm y agrega la dimensión dada."""
    if df.empty or "municipio_norm" not in df.columns:
        return pd.Series(dtype=int)
    if col_cantidad and col_cantidad in df.columns:
        serie = df.groupby("municipio_norm")[col_cantidad].sum()
    else:
        serie = df.groupby("municipio_norm").size()
    return serie.rename(label)


tablas_ivf = {}

# Dim 1 — Hurto
df_h = datos_limpios.get("hurto_policia", pd.DataFrame())
if not df_h.empty:
    t = agrupar_municipio(df_h, "cantidad", "hurto_total")
    if not t.empty:
        tablas_ivf["hurto_total"] = t
        print(f"  ✅ Dim 1 HURTO:              {len(t)} municipios")
else:
    print("  ⚠ Dim 1 HURTO: sin datos")

# Dim 2 — VIF (Policía + INMLCF + Fiscalía SPOA)
dfs_vif = []
for clave_vif, col_cant in [("vif_policia","cantidad"), ("vif_inmlcf",None)]:
    df_v = datos_limpios.get(clave_vif, pd.DataFrame())
    if not df_v.empty:
        dfs_vif.append(agrupar_municipio(df_v, col_cant))

df_fv = datos_limpios.get("fiscalia_vif", pd.DataFrame())
if not df_fv.empty:
    col_fv = "total_procesos" if "total_procesos" in df_fv.columns else None
    dfs_vif.append(agrupar_municipio(df_fv, col_fv))

if dfs_vif:
    vif_total = pd.concat(dfs_vif, axis=1).sum(axis=1).rename("vif_total").astype(int)
    tablas_ivf["vif_total"] = vif_total
    n_fuentes = len(dfs_vif)
    print(f"  ✅ Dim 2 VIF ({n_fuentes} fuentes):      {len(vif_total)} municipios")
else:
    print("  ⚠ Dim 2 VIF: sin datos")

# Dim 3 — Medidas ICBF
df_icbf = datos_limpios.get("comisarias_icbf", pd.DataFrame())
if not df_icbf.empty:
    if "municipio_norm" not in df_icbf.columns and "municipio" in df_icbf.columns:
        df_icbf["municipio_norm"] = df_icbf["municipio"].apply(normalizar_texto)
    t = agrupar_municipio(df_icbf, "cantidad" if "cantidad" in df_icbf.columns else None, "medidas_proteccion_total")
    if not t.empty:
        tablas_ivf["medidas_proteccion_total"] = t
        print(f"  ✅ Dim 3 ICBF:              {len(t)} municipios")
else:
    print("  ⚠ Dim 3 ICBF: sin datos")

# Dim 4 — Alimentos/Familia
df_rj = datos_limpios.get("rama_judicial", pd.DataFrame())
if not df_rj.empty:
    if "municipio_norm" not in df_rj.columns and "municipio" in df_rj.columns:
        df_rj["municipio_norm"] = df_rj["municipio"].apply(normalizar_texto)
    if "total_ingresos" in df_rj.columns:
        df_rj["total_ingresos"] = pd.to_numeric(df_rj["total_ingresos"], errors="coerce").fillna(0)
        t = agrupar_municipio(df_rj, "total_ingresos", "alimentos_familia_total")
    else:
        t = agrupar_municipio(df_rj, None, "alimentos_familia_total")
    if not t.empty:
        tablas_ivf["alimentos_familia_total"] = t
        print(f"  ✅ Dim 4 ALIMENTOS/FAMILIA: {len(t)} municipios")
else:
    print("  ⚠ Dim 4 ALIMENTOS/FAMILIA: sin datos")

# ── Construir tabla consolidada ───────────────────────────────────────────────
print()
if tablas_ivf:
    df_ivf = (
        pd.concat(tablas_ivf.values(), axis=1)
        .fillna(0).astype(int)
        .reset_index().rename(columns={"index":"municipio_norm"})
    )

    dims_disponibles = [c for c in
        ["hurto_total","vif_total","medidas_proteccion_total","alimentos_familia_total"]
        if c in df_ivf.columns]

    # Score bruto (v4 — solo para referencia comparativa)
    df_ivf["ivf_score_bruto"] = df_ivf[dims_disponibles].sum(axis=1)

    # ── Score PONDERADO (v5) ─────────────────────────────────────────────────
    # Normalizar cada dimensión (0→1) luego aplicar pesos.
    pesos_activos = {k: v for k, v in PESOS_IVF.items() if k in dims_disponibles}
    total_peso    = sum(pesos_activos.values())

    df_ivf["ivf_score_ponderado"] = 0.0
    for dim, peso in pesos_activos.items():
        col_max = df_ivf[dim].max()
        if col_max > 0:
            df_ivf["ivf_score_ponderado"] += (df_ivf[dim] / col_max) * (peso / total_peso)
    df_ivf["ivf_score_ponderado"] = (df_ivf["ivf_score_ponderado"] * 100).round(2)

    # ── Tasa PER CÁPITA por 100 000 hab. (v5) ────────────────────────────────
    # Corrige sesgo de densidad poblacional que sobreestimaba Cali en v4.
    df_ivf["poblacion_2024"] = df_ivf["municipio_norm"].map(DANE_POB_2024).fillna(0).astype(int)
    df_ivf["ivf_tasa_100k"]  = df_ivf.apply(
        lambda r: round(r["ivf_score_bruto"] / r["poblacion_2024"] * 100_000, 2)
        if r["poblacion_2024"] > 0 else None, axis=1
    )

    # Rankings comparativos
    df_ivf["rank_bruto"]      = df_ivf["ivf_score_bruto"].rank(ascending=False, method="min").astype(int)
    df_ivf["rank_ponderado"]  = df_ivf["ivf_score_ponderado"].rank(ascending=False, method="min").astype(int)
    df_ivf["rank_percapita"]  = df_ivf["ivf_tasa_100k"].rank(ascending=False, method="min", na_option="bottom").astype(int)

    df_ivf = df_ivf.sort_values("ivf_score_ponderado", ascending=False)

    print("=" * 72)
    print(f"TABLA IVF v5 — {len(dims_disponibles)}/4 dims activas")
    print("=" * 72)
    cols_show = (["municipio_norm"] + dims_disponibles +
                 ["ivf_score_ponderado","ivf_tasa_100k","rank_ponderado","rank_percapita"])
    cols_show = [c for c in cols_show if c in df_ivf.columns]
    print(df_ivf[cols_show].head(15).to_string(index=False))

    print("\n📌 rank_percapita != rank_ponderado → municipios pequeños con")
    print("   alta incidencia relativa aparecen arriba en la tasa per cápita.")

    ruta_ivf = os.path.join(OUTPUT_DIR, "lexdata_tabla_co_ocurrencia_IVF_v5.csv")
    df_ivf.to_csv(ruta_ivf, index=False, encoding="utf-8-sig")
    print(f"\n✅ Exportado → {ruta_ivf}")

    dims_faltantes = [d for d in
        ["hurto_total","vif_total","medidas_proteccion_total","alimentos_familia_total"]
        if d not in dims_disponibles]
    if dims_faltantes:
        print(f"   ⚠ Faltantes: {dims_faltantes}")
else:
    print("⚠  Sin datos suficientes para construir la tabla IVF.")

  ✅ Dim 1 HURTO:              42 municipios
  ⚠ Dim 2 VIF: sin datos suficientes
  ⚠ Dim 3 ICBF: DataFrame vacío (dataset no disponible)
  ⚠ Dim 4 ALIMENTOS/FAMILIA: DataFrame vacío (dataset no disponible)

TABLA CO-OCURRENCIA IVF — 1/4 dimensiones activas
     municipio_norm  hurto_total  ivf_score_provisional
               CALI       102211                 102211
            PALMIRA         9378                   9378
       BUENAVENTURA         3634                   3634
              TULUA         3207                   3207
            JAMUNDI         3176                   3176
              YUMBO         2989                   2989
GUADALAJARA DE BUGA         2093                   2093
         CANDELARIA         1750                   1750
            CARTAGO         1651                   1651
            FLORIDA          808                    808
         EL CERRITO          780                    780
            PRADERA          573                    573
            SEV

In [16]:
# ─────────────────────────────────────────────────────────
# RESUMEN EJECUTIVO DEL PIPELINE
# ─────────────────────────────────────────────────────────
print("\n" + "═" * 62)
print("RESUMEN PIPELINE — LexData Nicho Familiar v4")
print("═" * 62)

total_tx = 0
for clave, df in datos_limpios.items():
    n = len(df)
    estado = "✅" if n > 0 else "⚠️ "
    tipo = df["tipo_ciclo"].iloc[0] if n > 0 and "tipo_ciclo" in df.columns else "—"
    print(f"  {estado} {clave:<28} {n:>7,} registros  [{tipo}]")
    if n > 0 and clave != "comisarias_directorio":
        total_tx += n

print(f"\n  Total registros transaccionales: {total_tx:,}")
print(f"  Departamento(s) filtrado(s):     {DEPARTAMENTOS_FILTRO or 'TODOS'}")
print(f"  Rango de años:                   {min(YEARS)}–{max(YEARS)}")

# Coordenadas comisarías
df_geo = datos_limpios.get("comisarias_directorio", pd.DataFrame())
if not df_geo.empty and "lat" in df_geo.columns:
    n_geo = df_geo[["lat", "lon"]].notna().all(axis=1).sum()
    print(f"  Comisarías georreferenciadas:    {n_geo}/{len(df_geo)} (Valle del Cauca bbox)")

print(f"\n  Archivos generados en '{OUTPUT_DIR}':")
for f in sorted(os.listdir(OUTPUT_DIR)):
    ruta = os.path.join(OUTPUT_DIR, f)
    kb = os.path.getsize(ruta) / 1024
    print(f"    📄 {f:<50} {kb:>7.1f} KB")

print("\n" + "═" * 62)
print("Pipeline v4 completado.")
print("═" * 62)


══════════════════════════════════════════════════════════════
RESUMEN PIPELINE — LexData Nicho Familiar v4
══════════════════════════════════════════════════════════════
  ⚠️  rama_judicial                      0 registros  [—]
  ⚠️  comisarias_icbf                    0 registros  [—]
  ⚠️  vif_inmlcf                         0 registros  [—]
  ✅ hurto_policia                 17,408 registros  [HURTO]
  ✅ comisarias_directorio             60 registros  [COMISARIA_GEO]
  ⚠️  vif_policia                        0 registros  [—]

  Total registros transaccionales: 17,408
  Departamento(s) filtrado(s):     ['VALLE DEL CAUCA']
  Rango de años:                   2020–2024
  Comisarías georreferenciadas:    54/60 (Valle del Cauca bbox)

  Archivos generados en 'data_judicial':
    📄 lexdata_comisarias_directorio_geo.csv                 23.0 KB
    📄 lexdata_hurto_policia.csv                           3117.1 KB
    📄 lexdata_nicho_familiar_CONSOLIDADO.csv              3027.8 KB
    📄 lexdata_t

---
## Notas técnicas — v5

### Correcciones y mejoras aplicadas

| # | Problema (v4) | Solución (v5) |
|---|---|---|
| 1 | App Token comentado | Carga automática desde `env/.env` → `X-App-Token` |
| 2 | `socrata_get` sin retry | Backoff exponencial: 1s, 2s, 4s antes de rendirse |
| 3 | IVF = suma absoluta (sesgo poblacional) | Tasa por 100 000 hab. (DANE 2024) + score ponderado |
| 4 | Solo hurto en IVF | 4 dims + Fiscalía SPOA como 3ª fuente VIF |
| 5 | SICOF sin privacidad | SHA-256+salt pseudoanonimiza cédulas antes de persistir |
| 6 | YEARS hasta 2024 | Extendido a 2025 |

### Activar App Token Socrata
El token se carga desde `env/.env` automáticamente. Si `python-dotenv` no está instalado:
```bash
pip install python-dotenv
```
O exporta la variable de entorno manualmente antes de iniciar Jupyter:
```bash
export id_API="tu_token_aqui"
```

### Fuentes adicionales sugeridas para el Motor IVF

| Fuente | Dataset ID | Relevancia |
|---|---|---|
| Fiscalía — Víctimas VIF | `hdcg-8p5v` | Complemento ciclo VIF |
| ICBF — Restablecimiento derechos | `n5ps-mbf7` | Menores en el ciclo |
| SISBEN — Hogares vulnerables | `y8ca-4rkf` | IVF ↔ pobreza |
| Comisarías VIF Los Patios | `rg25-vxp5` | Dataset granular piloto |

### IVF: diferencia entre métricas v5

| Métrica | Qué mide | Cuándo usar |
|---|---|---|
| `ivf_score_bruto` | Volumen absoluto de eventos | Carga operativa de comisarías |
| `ivf_score_ponderado` | Intensidad relativa ponderada | Priorización interanual |
| `ivf_tasa_100k` | Incidencia per cápita | Comparación entre municipios de distinto tamaño |

> La tasa per cápita es la métrica estadísticamente válida para comparar municipios.
> El score ponderado es más útil para tracking temporal dentro del mismo municipio.

### Pipeline recomendado

1. Verificar tokens en `env/.env`
2. Ejecutar celda de diagnóstico — corregir IDs caídos con `buscar_dataset_id()`
3. Ejecutar `scrape_nicho_familiar(YEARS)`
4. Solicitar exportación SICOF a la entidad piloto → `ingestar_sicof()`
5. Fusionar público + SICOF en el DataFrame consolidado
6. Usar `lexdata_tabla_co_ocurrencia_IVF_v5.csv` como feature matrix inicial